In [1]:
# !pip install QuantumRingsLib

## Imports

In [2]:
from QuantumRingsLib import QuantumRingsProvider
from quantumrings.toolkit.qiskit import QrBackendV2

from qiskit import QuantumCircuit
from qiskit import transpile

import matplotlib.pyplot as plt
import numpy as np
import random
import math

from semiprimes import semiprimes

In [3]:
#testing the file access for the semiprimes is working
for bit_lengths, semiprime_numbers in semiprimes.items():
    print(f"Bit Length: {bit_lengths}, Number: {semiprime_numbers}")

Bit Length: 8, Number: 143
Bit Length: 22, Number: 3036893
Bit Length: 24, Number: 11426971
Bit Length: 26, Number: 58949987
Bit Length: 28, Number: 208241207
Bit Length: 30, Number: 857830637
Bit Length: 32, Number: 2776108693
Bit Length: 34, Number: 11455067797
Bit Length: 36, Number: 52734393667
Bit Length: 38, Number: 171913873883
Bit Length: 40, Number: 862463409547
Bit Length: 42, Number: 2830354423669
Bit Length: 44, Number: 12942106192073
Bit Length: 46, Number: 53454475917779
Bit Length: 48, Number: 255975740711783
Bit Length: 50, Number: 696252032788709
Bit Length: 52, Number: 3622511636491483
Bit Length: 54, Number: 15631190744806271
Bit Length: 56, Number: 51326462028714137
Bit Length: 58, Number: 217320198167105543
Bit Length: 60, Number: 827414216976034907
Bit Length: 62, Number: 3594396771839811733
Bit Length: 64, Number: 13489534701147995111
Bit Length: 66, Number: 48998116978431560767
Bit Length: 68, Number: 220295379750460962499
Bit Length: 70, Number: 757619317101213

## Shor Implementation

### Quantum Provider

In [ ]:
qr_provider = QuantumRingsProvider(token ='[redacted]', name='[redacted]')

### Modular Exponentiation

In [5]:
def mod_exp(a, N, qubits):
    # This function implements modular exponentiation as a quantum operation.
    # The function takes four parameters:
    # - `a`: The base of the modular exponentiation.
    # - `N`: The modulus for the operation.
    # - `qubits`: The number of qubits used in the quantum circuit.
    # The function creates a quantum circuit that applies Hadamard gates to all qubits to create superposition,
    # calculates the modular exponentiation for each qubit, and applies phase shifts based on the result.
    # It then converts the circuit into a controlled gate for use in quantum algorithms.
    
    U = QuantumCircuit(qubits)
    for i in range(qubits):
        U.h(i) 
    
    for j in range(qubits):
        a_mod_exp = pow(a, 2**j, N) 
        phase_value = 2 * math.pi * a_mod_exp / N 
        for q in range(qubits):
            U.p(phase_value, q)
    
    U = U.to_gate()
    U.name = f"{a}^x mod {N}"
    c_U = U.control()

    return c_U

### Inverse QFT

In [6]:
def inverse_qft(n_qubits):
    # This function implements the inverse Quantum Fourier Transform (QFT^-1).
    # The inverse QFT is the reverse of the QFT, which is a key component in many quantum algorithms.
    # The function creates a quantum circuit with `n` qubits and performs the following steps:
    # 1. Swaps the qubits to reverse their order, as QFT^-1 requires reversing the qubit order.
    # 2. Applies controlled phase gates (cp) to introduce phase shifts between qubits.
    # 3. Applies Hadamard gates (h) to each qubit to complete the inverse QFT transformation.
    
    qc = QuantumCircuit(n_qubits)

    for qubit in range(n_qubits//2):
        qc.swap(qubit, n_qubits-qubit-1)

    for i in range(n_qubits):
        for j in range(i):
            qc.cp(-math.pi/float(2**(i-j)), j, i)
        qc.h(i)

    qc.name = "QFT^-1"
    
    return qc

### Main Runtime

Note, all cells above still need to be run for the functions to work correctly in a jupyter notebook

In [ ]:
successful_factors = []
a_attempts_count = []
gate_count_list = []

for bit_lengths, semiprime_numbers in semiprimes.items():
    N = semiprime_numbers
    a_attempts = []
    while 1:
        p, q = 1, 1
        
        a = random.randint(2, N - 2)
        if math.gcd(a, N) != 1:
            p = math.gcd(a, N)
            q = N // p

        print("factoring", N, "with a =", a)
        if math.gcd(a, N) > 1:
            p = math.gcd(a, N)
            q = N // p
            break

        n_qubits = (N.bit_length() // 2)
        # n = math.ceil(math.log2(N))
        # source_qubits = n + 2  # Reduce from 2n to n+2 — works reasonably well
        # target_qubits = n      # Minimum needed to mod exp N

        # n_qubits = source_qubits + target_qubits
        # print("n_qubits", n_qubits)

        #initialization
        qc = QuantumCircuit(2 * n_qubits, n_qubits)
        
        for q in range(n_qubits):
            qc.h(q) 
        
        qc.x(n_qubits * 2 - 1)  #the last gate is switched to 1 


        #modular exponentiation
        for q in range(n_qubits):
            qc.append(mod_exp(a, N, n_qubits), [q] + [i + n_qubits for i in range(n_qubits)])


        #inverse QFT
        qc.append(inverse_qft(n_qubits), range(n_qubits))  
        qc.measure(range(n_qubits), range(n_qubits))  

        #backend
        backend = QrBackendV2(qr_provider, num_qubits=qc.num_qubits)
        qc_transpiled = transpile(qc, backend, initial_layout=[i for i in range(qc.num_qubits)])
        job = backend.run(qc_transpiled, shots=1000)
        result = job.result()

        counts = result.get_counts()

        gate_count = qc.size()
        gate_count_list.append(gate_count)

        #classical component
        r = int(max(counts, key=counts.get), 2)
        
        f1 = math.gcd(a**(r//2) - 1, N)
        f2 = math.gcd(a**(r//2) + 1, N)

        #check if the factors are valid
        if f1 == 1 and f2 > 1:
            p = f2
            q = N // f2
        elif f2 == 1 and f1 > 1:
            p = f1
            q = N // f1

        if p != 1 and q != 1 and p * q == N:
            break #found the factors

        a_attempts.append(a) #ignoring repeats so the length is not perfect if the random generator is bad 
        print(a_attempts)
    gate_count_list.append(gate_count)

    # ----------------------------------------------------------------
    
    print("Successfully factored", N, "into", p, "and", q, "using", gate_count * len(a_attempts), "gates")
    successful_factors.append(N)
    a_attempts_count.append(len(a_attempts))
    print()
    print()
    print()

factoring 143 with a = 140
n_qubits 4
[140]
factoring 143 with a = 63
n_qubits 4
[140, 63]
factoring 143 with a = 27
n_qubits 4
[140, 63, 27]
factoring 143 with a = 128
n_qubits 4
[140, 63, 27, 128]
factoring 143 with a = 120
n_qubits 4
Successfully factored 143 into 11 and 13 using 56 gates



factoring 3036893 with a = 150768
n_qubits 11
[150768]
factoring 3036893 with a = 137030
n_qubits 11
[150768, 137030]
factoring 3036893 with a = 2714151
n_qubits 11
[150768, 137030, 2714151]
factoring 3036893 with a = 600410
n_qubits 11
[150768, 137030, 2714151, 600410]
factoring 3036893 with a = 509501
n_qubits 11
[150768, 137030, 2714151, 600410, 509501]
factoring 3036893 with a = 113341
n_qubits 11
[150768, 137030, 2714151, 600410, 509501, 113341]
factoring 3036893 with a = 1138217
n_qubits 11
[150768, 137030, 2714151, 600410, 509501, 113341, 1138217]
factoring 3036893 with a = 729639
n_qubits 11
[150768, 137030, 2714151, 600410, 509501, 113341, 1138217, 729639]
factoring 3036893 with a = 993

KeyboardInterrupt: 

In [8]:
for i, N in enumerate(successful_factors):
    print("Successfully factored", N, "in", a_attempts_count[i], "quantum attempts, for a total of", gate_count_list[i] * a_attempts_count[i], "quantum gates")

Successfully factored 143 in 4 quantum attempts, for a total of 56 quantum gates


-------------